In [2]:
# Resumes --> Chunks --> embeddings (Sparse & Dense) --> Metadata --> Pincecone (Up-sert)

In [3]:
# Cell 1: Install (if needed) + Imports

# If running fresh, uncomment:
# !pip install langchain-community langchain-huggingface langchain-google-genai \
#             sentence-transformers pinecone beautifulsoup4 scikit-learn

from pathlib import Path
from typing import List, Dict, Any
import os
import json
import pickle

from pinecone import Pinecone, ServerlessSpec

from sklearn.feature_extraction.text import TfidfVectorizer

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from bs4 import BeautifulSoup


# Config

In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

# openai_api_key = os.getenv("OPENAI_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

In [15]:
# from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

## Define resume path
RESUME_DIR = Path("resume_dir")

## Define an Index name
RESUME_INDEX_NAME = "resume-hybrid-index"

# Embeddings model
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


# LLM for parsing resumes into structured JSON
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    api_key = gemini_api_key,
)

# llm = ChatOpenAI(
#     model="gpt-4.1",
#     api_key= openai_api_key
# )

# Creating Index

In [16]:
# Step 1
pc = Pinecone(api_key=pinecone_api_key)

if RESUME_INDEX_NAME not in [idx["name"] for idx in pc.list_indexes()]:
    pc.create_index(
        name=RESUME_INDEX_NAME,
        dimension=384,            # must match dense model
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    
index = pc.Index(RESUME_INDEX_NAME)

In [17]:
index

Index(host='https://resume-hybrid-index-503dlyx.svc.aped-4627-b74a.pinecone.io')

# Loading the resume data

In [18]:
# Loading the pdf files from location # split or chunks the info present
def load_resume_text(path:Path):
    suffix = path.suffix.lower()
    if(suffix == '.pdf'):
        loader = PyPDFLoader(str(path))
        docs = loader.load()
        return "\n".join(d.page_content for d in docs)

In [19]:
#Testing 
test_path = Path("resume_dir/Ashley_Phillips_Resume_32.pdf")
test_path

WindowsPath('resume_dir/Ashley_Phillips_Resume_32.pdf')

In [20]:
resume_text = load_resume_text(test_path)
resume_text

'ASHLEY PHILLIPS\nStaff Accountant\nContact Information:\nEmail: ashley.phillips@email.com\nPhone: (599) 609-6545\nLocation: Memphis, TN\nLinkedIn: linkedin.com/in/ashleyphillips\nPROFESSIONAL SUMMARY\nExperienced accounting professional with 3+ years of progressive experience in financial reporting,\nanalysis, and compliance. Proven track record of improving processes and delivering accurate financial\ninformation. Strong expertise in accounting principles and software applications.\nTECHNICAL SKILLS\n\x7f Variance Analysis\n\x7f Trend Analysis\n\x7f Portfolio Management\n\x7f KPI Development\n\x7f Data Analysis\n\x7f Forecasting\n\x7f Due Diligence\n\x7f Financial Modeling\nSOFTWARE PROFICIENCY\n\x7f Microsoft Excel\n\x7f Database Management\n\x7f NetSuite\n\x7f Financial Software\n\x7f Python\n\x7f SAP\nPROFESSIONAL EXPERIENCE\nCompliance Manager | Professional Accounting Partners | 2021 - Present\n\x7f Led financial reporting and analysis for $50M+ revenue company\n\x7f Managed mon

In [21]:
# Chuncking (Based on the LLM)
## --> Metadata 
########## --> Skills, User detailes (location, name, email), ID, Summary , Project Experience
def llm_parse_resume(raw_text):
    prompt = f"""
        You are a strict JSON resume parser.
        
        Return ONLY valid minified JSON. No markdown, no commentary.
        In the summary generate a summary of his experience and projects
        Schema (use exactly these keys):
        {{
          "summary": "string",
          "skills": ["string", ...],
          "CERTIFICATIONS" : ["string",...],
          "user details" : [{{
              "email" : ["string"],
              "location" : ["string"],
          }}],
          "email" : ["string"],
          "Location" : ["string"],
          "experiences": [
            {{
              "title": "string",
              "company": "string",
              "location": "string",
              "start_date": "string",
              "end_date": "string",
              "description": "string",
              "skills": ["string", ...]
            }}
          ],
          "education": [
            {{
              "degree": "string",
              "institution": "string",
              "year": "string"
            }}
          ],
          "projects": [
            {{
              "name": "string",
              "description": "string",
              "skills": ["string", ...]
            }}
          ]
        }}
        
        If something is missing, use "" or [], don't invent new text other than the existing resume info
        
        Resume:
        \"\"\"{raw_text[:12000]}\"\"\"
    """.strip()
    
    resp = llm.invoke(prompt)
    content = getattr(resp, "content", resp)
    return json.loads(content)

In [22]:
llm_parserd_text = llm_parse_resume(resume_text)
llm_parserd_text

{'summary': 'Ashley Phillips is an experienced Staff Accountant with over 3 years of progressive experience in financial reporting, analysis, and compliance. As a Compliance Manager at Professional Accounting Partners, she led financial reporting for a $50M+ revenue company, managed month-end closing, and implemented process improvements resulting in 20% efficiency gains. She also has a strong background in accounting principles and software, including Microsoft Excel, NetSuite, Python, and SAP.',
 'skills': ['Variance Analysis',
  'Trend Analysis',
  'Portfolio Management',
  'KPI Development',
  'Data Analysis',
  'Forecasting',
  'Due Diligence',
  'Financial Modeling',
  'Microsoft Excel',
  'Database Management',
  'NetSuite',
  'Financial Software',
  'Python',
  'SAP'],
 'CERTIFICATIONS': ['CIA', 'Enrolled Agent'],
 'user details': [{'email': ['ashley.phillips@email.com'],
   'location': ['Memphis, TN']}],
 'email': ['ashley.phillips@email.com'],
 'Location': ['Memphis, TN'],
 '

In [23]:
Metadata --> Skills set, Certification, years of experience and education, location
Embeddings --> Summary, Skills, years of experience, location...

SyntaxError: invalid syntax (2617089908.py, line 1)

In [24]:

embeddings_text = llm_parserd_text.get("summary") + str(llm_parserd_text.get("skills")) + str(llm_parserd_text.get("Location"))
embeddings_text

"Ashley Phillips is an experienced Staff Accountant with over 3 years of progressive experience in financial reporting, analysis, and compliance. As a Compliance Manager at Professional Accounting Partners, she led financial reporting for a $50M+ revenue company, managed month-end closing, and implemented process improvements resulting in 20% efficiency gains. She also has a strong background in accounting principles and software, including Microsoft Excel, NetSuite, Python, and SAP.['Variance Analysis', 'Trend Analysis', 'Portfolio Management', 'KPI Development', 'Data Analysis', 'Forecasting', 'Due Diligence', 'Financial Modeling', 'Microsoft Excel', 'Database Management', 'NetSuite', 'Financial Software', 'Python', 'SAP']['Memphis, TN']"

In [25]:
def build_resume_doc(parsed_text, resume_id, filename):
    overall_text = []
    summary = parsed_text.get("summary")
    skills = parsed_text.get("skills")
    experiences = parsed_text.get("experiences")
    education = parsed_text.get("education")
    CERTIFICATIONS =  parsed_text.get("CERTIFICATIONS")
    projects =  parsed_text.get("projects")
    location =  parsed_text.get("Location")
    # For Emebedding part
    if summary: 
        overall_text.append(f"Summary: {summary}")
    if skills: 
        overall_text.append("Skills: " + ", ".join(skills))
    
    full_text = "\n".join(overall_text).strip()

    # Metadata part
    all_skills = set()
    for s in skills:
        all_skills.add(s.strip())
    roles = set()
    companies = set()
    for e in experiences:
        roles.add((e.get("title") or "").strip())
        companies.add((e.get("company") or "").strip())
    metadata = {"resume_id": resume_id, "filename": filename, "skills": sorted(all_skills), "roles": sorted(roles), "companies": sorted(companies), "location" : sorted(location)}
    
    return Document(page_content = full_text, metadata = metadata)

In [26]:
ingest_data = build_resume_doc(llm_parserd_text, 1, "Ashley_Phillips_Resume_32.pdf")

In [27]:
ingest_data

Document(metadata={'resume_id': 1, 'filename': 'Ashley_Phillips_Resume_32.pdf', 'skills': ['Data Analysis', 'Database Management', 'Due Diligence', 'Financial Modeling', 'Financial Software', 'Forecasting', 'KPI Development', 'Microsoft Excel', 'NetSuite', 'Portfolio Management', 'Python', 'SAP', 'Trend Analysis', 'Variance Analysis'], 'roles': ['Compliance Manager'], 'companies': ['Professional Accounting Partners'], 'location': ['Memphis, TN']}, page_content='Summary: Ashley Phillips is an experienced Staff Accountant with over 3 years of progressive experience in financial reporting, analysis, and compliance. As a Compliance Manager at Professional Accounting Partners, she led financial reporting for a $50M+ revenue company, managed month-end closing, and implemented process improvements resulting in 20% efficiency gains. She also has a strong background in accounting principles and software, including Microsoft Excel, NetSuite, Python, and SAP.\nSkills: Variance Analysis, Trend Ana

In [28]:
### Step 4: Chunk and split for all the resumes in the document folder
def step4_load_and_split(cfg):
    resume_dir = cfg['resume_dir']
    docs = []
    for fp in resume_dir.iterdir():
        print(fp)
        raw  = load_resume_text(fp).strip()   ## Convert pdf to text
        resume_id = fp.stem
        parsed = llm_parse_resume(raw)  ## Text to into format required using LLM
        doc = build_resume_doc(parsed, resume_id, fp.name)  # Embedding text and Metadata filter 
        docs.append(doc)
    return {"docs": docs}

In [29]:
payload = step4_load_and_split({"resume_dir" : RESUME_DIR})

resume_dir\Andrew_Green_Resume_27.pdf
resume_dir\Angela_Lewis_Resume_09.pdf
resume_dir\Ashley_Phillips_Resume_32.pdf


In [30]:
payload

{'docs': [Document(metadata={'resume_id': 'Andrew_Green_Resume_27', 'filename': 'Andrew_Green_Resume_27.pdf', 'skills': ['Account Analysis', 'Accounts Payable', 'Adobe Acrobat', 'Bank Reconciliation', 'Financial Reporting', 'Fixed Asset Management', 'General Ledger Management', 'Journal Entry Preparation', 'Microsoft Excel', 'Month-End Closing Procedures', 'Outlook', 'Payroll Processing', 'PowerPoint', 'SQL', 'Sage', 'Tax Preparation', 'Variance Analysis', 'Year-End Closing'], 'roles': ['Accounting Intern'], 'companies': ['Premier Financial Advisors'], 'location': ['Long Beach, CA']}, page_content='Summary: Recent accounting graduate with a strong academic foundation from Columbia University and an internship at Premier Financial Advisors. His experience includes assisting with month-end closing, processing accounts payable, performing bank reconciliations, maintaining general ledger accounts, and creating Excel spreadsheets for financial analysis. He is eager to begin a career focusin

In [31]:
VECTORIZER_PATH = "tfidf_vectorizer.pkl"

# Creating embeddings
def step5_encode(payload):
    # Creating Dense embeddings
    docs = payload["docs"]
    corpus = [d.page_content for d in docs]
    embed = HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        encode_kwargs={"normalize_embeddings": True},
    )
    dense_vectors = embed.embed_documents(corpus)

    #Creating Sparse embeddings
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=1,
    )
    tfidf_matrix = vectorizer.fit_transform(corpus)

    # Save vectorizer
    with open(VECTORIZER_PATH, "wb") as f:
        pickle.dump(vectorizer, f)

    print("TF-IDF vectorizer saved!")
    
    return {"docs": docs, "dense_vectors": dense_vectors, "tfidf_matrix":tfidf_matrix}

In [32]:
encoded_payload = step5_encode(payload)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

TF-IDF vectorizer saved!


In [34]:
encoded_payload

{'docs': [Document(metadata={'resume_id': 'Andrew_Green_Resume_27', 'filename': 'Andrew_Green_Resume_27.pdf', 'skills': ['Account Analysis', 'Accounts Payable', 'Adobe Acrobat', 'Bank Reconciliation', 'Financial Reporting', 'Fixed Asset Management', 'General Ledger Management', 'Journal Entry Preparation', 'Microsoft Excel', 'Month-End Closing Procedures', 'Outlook', 'Payroll Processing', 'PowerPoint', 'SQL', 'Sage', 'Tax Preparation', 'Variance Analysis', 'Year-End Closing'], 'roles': ['Accounting Intern'], 'companies': ['Premier Financial Advisors'], 'location': ['Long Beach, CA']}, page_content='Summary: Recent accounting graduate with a strong academic foundation from Columbia University and an internship at Premier Financial Advisors. His experience includes assisting with month-end closing, processing accounts payable, performing bank reconciliations, maintaining general ledger accounts, and creating Excel spreadsheets for financial analysis. He is eager to begin a career focusin

In [35]:
def csr_row_to_pinecone_sparse(csr_row) -> Dict[str, List[float]]:
    coo = csr_row.tocoo()
    return {
        "indices": coo.col.tolist(),
        "values": coo.data.astype(float).tolist()
    }

def step6_package_and_upsert(payload: Dict[str, Any]) -> Dict[str, Any]:
    docs: List[Document] = payload["docs"]
    dense_vectors = payload.get("dense_vectors", [])
    tfidf_matrix = payload.get("tfidf_matrix")

    if not docs or tfidf_matrix is None:
        return {"upserted": 0, "index": RESUME_INDEX_NAME}

    vectors = []

    for i, (doc, dense) in enumerate(zip(docs, dense_vectors)):
        sparse = csr_row_to_pinecone_sparse(tfidf_matrix[i])
        resume_id = doc.metadata.get("resume_id")
        vid = resume_id
        meta = {
            **{k: v for k, v in doc.metadata.items() if k != "text"},
            "text": doc.page_content[:1200],  # snippet for preview
        }
        vectors.append(
            {
                "id": vid,
                "values": dense,
                "sparse_values": sparse,
                "metadata": meta,
            }
        )
    if vectors:
        index.upsert(vectors=vectors)
    return {"upserted": len(vectors), index: RESUME_INDEX_NAME}

In [36]:
step6_package_and_upsert(encoded_payload)

{'upserted': 3,
 Index(host='https://resume-hybrid-index-503dlyx.svc.aped-4627-b74a.pinecone.io'): 'resume-hybrid-index'}